# Assignment 18: Text Vectorization Techniques
### Author: Parth Dadhaniya

**Problem Statement:**
Convert cleaned text into numerical vector representations for NLP.
Techniques implemented:
- One-Hot Encoding (Manual & Scikit-Learn)
- Bag of Words (BoW) & Word Frequency
- N-grams (Unigrams, Bigrams, Trigrams, Combined)
- TF-IDF Vectorization & Comparison with BoW
- Vectorizer Parameter Exploration

**Dataset:** `cleaned_text_dataset.csv` containing 25 preprocessed text samples.

## PART 1 — One-Hot Encoding (Text Level)
### Task 1: Manual One-Hot Encoding

In [ ]:
import pandas as pd

df = pd.read_csv('cleaned_text_dataset.csv')
sentences = df['final_clean_text'].head(5).tolist()

# build vocabulary manually
vocab = sorted(list(set(' '.join(sentences).split())))
print('Vocabulary Size:', len(vocab))

# build one-hot vectors manually
manual_matrix = []
for s in sentences:
    words = set(s.split())
    vec = [1 if w in words else 0 for w in vocab]
    manual_matrix.append(vec)

df_manual = pd.DataFrame(manual_matrix, columns=vocab, index=[f'Sentence {i+1}' for i in range(5)])
print('\nOne-Hot Matrix (First 6 Columns):')
print(df_manual.iloc[:, :6])

### Task 2: One-Hot Encoding Using Scikit-Learn

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

cv_binary = CountVectorizer(binary=True)
X_binary = cv_binary.fit_transform(sentences)

print('Vocabulary Size:', len(cv_binary.vocabulary_))
df_sklearn = pd.DataFrame(
    X_binary.toarray(),
    columns=cv_binary.get_feature_names_out(),
    index=[f'Sentence {i+1}' for i in range(5)]
)
print('\nFirst 6 Columns of Scikit-Learn Encoded Matrix:')
print(df_sklearn.iloc[:, :6])

## PART 2 — Bag of Words (BoW)
### Task 3: Bag of Words Representation

In [ ]:
corpus = df['final_clean_text'].tolist()
cv = CountVectorizer()
X_bow = cv.fit_transform(corpus)
feature_names = cv.get_feature_names_out()

print('Total Documents:     ', len(corpus))
print('Total Vocabulary Size:', len(feature_names))
print('BoW Matrix Shape:    ', X_bow.shape)

# sample feature vector for Document 1
print('\nNon-zero features in Document 1:')
doc1 = X_bow[0].toarray()[0]
for word, count in zip(feature_names, doc1):
    if count > 0:
        print(f'  {word} : {count}')

### Task 4: Understanding Word Frequency

In [ ]:
totals = X_bow.toarray().sum(axis=0)
word_freq = pd.Series(totals, index=feature_names).sort_values(ascending=False)

print('Top 10 Most Frequent Words:')
for w, c in word_freq.head(10).items():
    print(f'  {w:<12} -> {c} times')

print('\nLeast Frequent Words (Count = 1):', (word_freq == 1).sum())
print('Sample least frequent:', list(word_freq[word_freq == 1].head(5).index))

## PART 3 — N-Grams
### Task 5: Unigrams, Bigrams & Trigrams

In [ ]:
cv_uni = CountVectorizer(ngram_range=(1, 1)).fit(corpus)
cv_bi = CountVectorizer(ngram_range=(2, 2)).fit(corpus)
cv_tri = CountVectorizer(ngram_range=(3, 3)).fit(corpus)

print(f'Unigrams (1, 1) Vocabulary: {len(cv_uni.get_feature_names_out())}')
print(f'Bigrams  (2, 2) Vocabulary: {len(cv_bi.get_feature_names_out())}')
print(f'Trigrams (3, 3) Vocabulary: {len(cv_tri.get_feature_names_out())}')

print('\nSample Bigrams: ', list(cv_bi.get_feature_names_out()[:5]))
print('Sample Trigrams:', list(cv_tri.get_feature_names_out()[:5]))

### Task 6: Combined N-Grams

In [ ]:
cv_comb = CountVectorizer(ngram_range=(1, 2)).fit(corpus)
print('Combined (1, 2) Vocabulary Size:', len(cv_comb.get_feature_names_out()))

features = cv_comb.get_feature_names_out()
bigrams = [f for f in features if ' ' in f]
print('\nSample Contextual Bigrams in Combined Model:')
print(bigrams[:6])

## PART 4 — TF-IDF Vectorization
### Task 7: TF-IDF Implementation

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)
tfidf_features = tfidf.get_feature_names_out()

print('TF-IDF Matrix Shape:', X_tfidf.shape)
print('Total Features:     ', len(tfidf_features))

doc1_tfidf = X_tfidf[0].toarray()[0]
print('\nDocument 1 TF-IDF Weights:')
for word, score in zip(tfidf_features, doc1_tfidf):
    if score > 0:
        print(f'  {word} : {round(score, 4)}')

### Task 8: BoW vs TF-IDF Comparison

In [ ]:
avg_tfidf = pd.Series(X_tfidf.toarray().mean(axis=0), index=tfidf_features).sort_values(ascending=False)

print('Top 5 Words with Highest Average TF-IDF:')
for w, score in avg_tfidf.head(5).items():
    print(f'  {w:<12} -> {round(score, 4)}')

print('\nWhy TF-IDF Down-Weights Common Words:')
print('- IDF = log(N / df). If a word appears in almost all documents, its IDF is near zero.')
print('- This ensures common uninformative words receive low weight while unique keywords stand out.')

## PART 5 — Practical Insights
### Task 9: Vectorizer Parameter Exploration

In [ ]:
print('Vocabulary Size under Different Settings:')
for params, label in [
    ({}, 'Baseline (Default)'),
    ({'max_features': 30}, 'max_features = 30'),
    ({'min_df': 2}, 'min_df = 2'),
    ({'max_df': 0.5}, 'max_df = 0.5')
]:
    v = TfidfVectorizer(**params).fit(corpus)
    print(f'  {label:<25} -> {len(v.get_feature_names_out())} words')

### Task 10: Conceptual Questions

1. **One-Hot Encoding vs. Bag of Words (BoW):**
   - One-Hot Encoding records binary presence (0 or 1).
   - Bag of Words records integer frequency counts of words in each sentence.

2. **Why N-Grams increase dimensionality:**
   - Bigrams and trigrams create combinations of adjacent words ($V^2, V^3$), rapidly increasing feature count.

3. **When to prefer TF-IDF over BoW:**
   - When building classifiers, search engines, or clustering models where common words would otherwise dominate counts.

4. **Limitations of count-based vectorization:**
   - Word Order: Ignores sentence grammar and structure.
   - Sparsity: Large matrices filled mostly with zeros.
   - Semantic Meaning: Synonyms ('car' vs 'automobile') are treated as completely independent dimensions.